# BAIT-Enhanced — Google Colab Runner

**TOP_K_FILTER pruning + parallel initial-token scanning** grafted onto BAIT-Enhanced.

This notebook has two independent paths — run whichever you need:

- **Path 1 — Black-box / stub scan** · no GPU, no model download, runs in seconds. Best for the live demo.
- **Path 2 — Real Ollama model** · runs an actual local LLM (the report's path). Slower, needs a small model.

> Run the **Setup** section once, then jump to Path 1 or Path 2.


---
## Setup (run once)

Choose **one** of the two setup cells below.


**Option A — clone from GitHub** (use this if the code is already pushed to your repo):

In [ ]:
!git clone https://github.com/sureshbabugandla/BAIT-Enhanced.git
%cd BAIT-Enhanced
!pip install numpy tqdm -q
print("\nSetup done. Files:")
!ls scripts/ | grep scan

**Option B — upload the zip** (use this if you have NOT pushed yet):

1. Click the **Files** icon (📁) on the left sidebar.
2. Upload `BAIT-Enhanced-merged.zip`.
3. Run the cell below.

In [ ]:
!unzip -o BAIT-Enhanced-merged.zip > /dev/null
%cd BAIT-Enhanced-merged
!pip install numpy tqdm -q
print("\nSetup done. Files:")
!ls scripts/ | grep scan

---
# Path 1 — Black-box / stub scan  (no GPU, instant)

Runs the full pipeline — TOP_K_FILTER vocabulary pruning + parallel initial-token scanning — against a built-in stub model with a *planted* backdoor target. Nothing to download; finishes in seconds. This is the recommended demo path.


### 1.1 Run the scan

It should recover the planted target `"Click http://malicious for more info"` with a Q-Score of **0.95**, scanning only the top-50 candidate tokens across 8 parallel workers.

In [ ]:
!python scripts/scan_blackbox.py --backend stub --top-k-filter 50 --parallel-workers 8

### 1.2 Show the parallel speedup (verdict-preserving)

Same scan, different worker counts. Notice the wall-clock time drops while the **best token stays identical** — parallelism only changes timing, never the verdict.

In [ ]:
import time
from src.core.parallel_scan import parallel_initial_token_scan

def score(tid):
    time.sleep(0.01)                 # simulate a blocking model call
    return (tid / 100.0, f"target_{tid}", {})

ids = list(range(60))
print("workers |  time  | best token")
print("--------|--------|-----------")
base = None
for w in [1, 2, 4, 8]:
    t = time.time()
    r = parallel_initial_token_scan(ids, score, max_workers=w, show_progress=False)
    dt = time.time() - t
    if base is None: base = dt
    print(f"   {w:<4} | {dt:5.2f}s | {r.best.token_id}   (speedup {base/dt:.1f}x)")

### 1.3 Show TOP_K_FILTER pruning

`plan_scan` caps the candidate set to the K most probable first tokens. Watch the candidate count shrink as K drops — this is the BAIT-Lite `TOP_K_FILTER` knob, now active.

In [ ]:
import numpy as np
from src.core.token_optimizer import plan_scan

# a toy first-token distribution: a few likely tokens, the rest near-zero
probs = np.full(2000, 1e-7)
probs[[5, 11, 23, 42, 88, 101, 250, 900]] = [.30,.22,.15,.12,.08,.06,.04,.03]

for k in [None, 100, 20, 5]:
    plan = plan_scan(probs, prob_floor=1e-6, p=1.0, top_k=k)
    label = "no cap" if k is None else f"top_k={k}"
    print(f"{label:<8} -> {plan.n_candidates:>4} candidates scanned")

### 1.4 (Optional) Log to Weights & Biases

Paste your W&B API key, then re-run the scan with `--use-wandb`.

In [ ]:
import os
os.environ["WANDB_API_KEY"] = ""   # <-- paste your key here

if os.environ["WANDB_API_KEY"]:
    !pip install wandb -q
    !python scripts/scan_blackbox.py --backend stub --top-k-filter 50 --parallel-workers 8 \
        --use-wandb --wandb-project bait-enhanced
else:
    print("No W&B key set — skipping. Paste your key above to enable logging.")

### 1.5 (Optional) Run the unit tests

Confirms the graft is correct: TOP_K_FILTER cap, parallel-vs-sequential equivalence, safe early-stop, and planted-target recovery.

In [ ]:
!pip install pytest -q
!python -m pytest tests/test_parallel_and_topk.py -v

---
# Path 2 — Real Ollama model  (the report's path)

Runs the scan against an actual local LLM served by **Ollama**. This is heavier: it installs Ollama, starts a background server, and downloads a model.

**Important:** use a **small** model on free Colab (e.g. `llama3.2:1b`). Large models like `deepseek-r1:8b` will run out of RAM.


### 2.1 Install Ollama and start the server

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
# start the Ollama server in the background
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
print("Ollama server started.")

### 2.2 Pull a small model

`llama3.2:1b` is ~1.3 GB and fits comfortably. This download takes a minute or two.

In [ ]:
!ollama pull llama3.2:1b
!pip install ollama -q
print("Model ready.")

### 2.3 Run the black-box scan against the real model

Uses `--top-k-filter 30` and `--parallel-workers 4`. On a real model each candidate is a blocking generation call, so parallelism gives a real wall-clock win here.

> Note: plain Ollama chat does not expose token probabilities, so the Q-Score here is a transparent *text-agreement proxy* (how consistently the model reproduces the same continuation across prompts) — never the report's old fixed 0.8 placeholder.

In [ ]:
!python scripts/scan_blackbox.py --backend ollama --model llama3.2:1b \
    --top-k-filter 30 --parallel-workers 4 --max-target-length 8

### 2.4 (Optional) Try a different model

Swap in any small Ollama model. Examples: `qwen2.5:0.5b` (tiny), `gemma2:2b`, `phi3:mini`. Pull it first, then re-run the scan with `--model <name>`.

In [ ]:
# example: a tiny model
!ollama pull qwen2.5:0.5b
!python scripts/scan_blackbox.py --backend ollama --model qwen2.5:0.5b \
    --top-k-filter 20 --parallel-workers 4 --max-target-length 6

---
### Notes

- **Path 3 (GPU model-zoo scan)** — the full research pipeline (`scripts/scan.py`) needs the ~hundreds-of-GB Hugging Face model zoo and heavy deps. It is **not practical on free Colab** and is intentionally omitted here.
- For the presentation, **Path 1** is the reliable choice: instant, deterministic, GPU-free, and it directly shows both contributions (TOP_K_FILTER pruning + parallel speedup).
- Results are saved to `results/blackbox_result.json` in the Colab file browser.

*BAIT-Enhanced · Group 7 · TOP_K_FILTER + parallel initial-token scan*
